https://docs.sqlalchemy.org/en/20/tutorial/index.html

In [1]:
import sqlalchemy
print(sqlalchemy.__version__)

2.0.45


In [3]:
from sqlalchemy import create_engine
engine = create_engine("sqlite+pysqlite:///:memory:", echo=True)

In [4]:
from sqlalchemy import text

with engine.connect() as conn:
    result = conn.execute(text("SELECT 'hello world'"))
    print(result.all())

2026-01-20 14:22:19,112 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 14:22:19,113 INFO sqlalchemy.engine.Engine SELECT 'hello world'
2026-01-20 14:22:19,114 INFO sqlalchemy.engine.Engine [generated in 0.00135s] ()
[('hello world',)]
2026-01-20 14:22:19,114 INFO sqlalchemy.engine.Engine ROLLBACK


In [5]:
with engine.connect() as conn:
    conn.execute(text("CREATE TABLE some_table (x int, y int)"))
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 1, "y": 1}, {"x": 2, "y": 4}],
    )
    conn.commit()

2026-01-20 14:26:30,403 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 14:26:30,403 INFO sqlalchemy.engine.Engine CREATE TABLE some_table (x int, y int)
2026-01-20 14:26:30,404 INFO sqlalchemy.engine.Engine [generated in 0.00130s] ()
2026-01-20 14:26:30,405 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-20 14:26:30,406 INFO sqlalchemy.engine.Engine [generated in 0.00089s] [(1, 1), (2, 4)]
2026-01-20 14:26:30,407 INFO sqlalchemy.engine.Engine COMMIT


In [6]:
with engine.begin() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 6, "y": 8}, {"x": 9, "y": 10}],
    )

2026-01-20 14:56:31,053 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 14:56:31,056 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-20 14:56:31,057 INFO sqlalchemy.engine.Engine [cached since 1814s ago] [(6, 8), (9, 10)]
2026-01-20 14:56:31,058 INFO sqlalchemy.engine.Engine COMMIT


In [7]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table"))
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-20 14:59:12,712 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 14:59:12,713 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table
2026-01-20 14:59:12,714 INFO sqlalchemy.engine.Engine [generated in 0.00265s] ()
x: 1 y: 1
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10
2026-01-20 14:59:12,716 INFO sqlalchemy.engine.Engine ROLLBACK


In [8]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y FROM some_table"))
    for dict_row in result.mappings():
        print(f"x: {dict_row["x"]} y: {dict_row["y"]}")

2026-01-20 15:02:40,738 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 15:02:40,740 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table
2026-01-20 15:02:40,741 INFO sqlalchemy.engine.Engine [cached since 211.2s ago] ()
x: 1 y: 1
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10
2026-01-20 15:02:40,744 INFO sqlalchemy.engine.Engine ROLLBACK


In [9]:
with engine.connect() as conn:
    result = conn.execute(text("SELECT x, y from some_table WHERE y > :y"), {"y": 2})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-20 15:24:20,042 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 15:24:20,044 INFO sqlalchemy.engine.Engine SELECT x, y from some_table WHERE y > ?
2026-01-20 15:24:20,045 INFO sqlalchemy.engine.Engine [generated in 0.00315s] (2,)
x: 2 y: 4
x: 6 y: 8
x: 9 y: 10
2026-01-20 15:24:20,047 INFO sqlalchemy.engine.Engine ROLLBACK


In [10]:
with engine.connect() as conn:
    conn.execute(
        text("INSERT INTO some_table (x, y) VALUES (:x, :y)"),
        [{"x": 11, "y": 12}, {"x": 13, "y": 14}],
    )
    conn.commit()

2026-01-20 15:28:18,101 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 15:28:18,103 INFO sqlalchemy.engine.Engine INSERT INTO some_table (x, y) VALUES (?, ?)
2026-01-20 15:28:18,104 INFO sqlalchemy.engine.Engine [cached since 3756s ago] [(11, 12), (13, 14)]
2026-01-20 15:28:18,106 INFO sqlalchemy.engine.Engine COMMIT


In [12]:
from sqlalchemy.orm import Session

stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-20 15:32:29,499 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 15:32:29,501 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-20 15:32:29,502 INFO sqlalchemy.engine.Engine [generated in 0.00091s] (6,)
x: 6 y: 8
x: 9 y: 10
x: 11 y: 12
x: 13 y: 14
2026-01-20 15:32:29,505 INFO sqlalchemy.engine.Engine ROLLBACK


In [13]:
with Session(engine) as session:
    result = session.execute(
        text("UPDATE some_table SET y=:y WHERE x=:x"),
        [{"x": 9, "y": 11}, {"x": 13, "y": 15}]
    )
    session.commit()

2026-01-20 15:34:29,143 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 15:34:29,145 INFO sqlalchemy.engine.Engine UPDATE some_table SET y=? WHERE x=?
2026-01-20 15:34:29,146 INFO sqlalchemy.engine.Engine [generated in 0.00119s] [(11, 9), (15, 13)]
2026-01-20 15:34:29,149 INFO sqlalchemy.engine.Engine COMMIT


In [14]:
from sqlalchemy.orm import Session

stmt = text("SELECT x, y FROM some_table WHERE y > :y ORDER BY x, y")
with Session(engine) as session:
    result = session.execute(stmt, {"y": 6})
    for row in result:
        print(f"x: {row.x} y: {row.y}")

2026-01-20 15:34:53,489 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 15:34:53,492 INFO sqlalchemy.engine.Engine SELECT x, y FROM some_table WHERE y > ? ORDER BY x, y
2026-01-20 15:34:53,493 INFO sqlalchemy.engine.Engine [cached since 147.1s ago] (6,)
x: 6 y: 8
x: 9 y: 11
x: 11 y: 12
x: 13 y: 15
2026-01-20 15:34:53,497 INFO sqlalchemy.engine.Engine ROLLBACK


In [15]:
from sqlalchemy import MetaData
metadata_obj = MetaData()

In [17]:
from sqlalchemy import Table, Column, Integer, String
user_table = Table(
    "user_account",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("name", String(30)),
    Column("fullname", String),
)

With the above example, when we wish to write code that refers to the `user_account` table in the database, we will use the `user_table` Python variable to refer to it.

In [18]:
user_table.c.name

Column('name', String(length=30), table=<user_account>)

In [19]:
user_table.c.keys()

['id', 'name', 'fullname']

The reference and API documentation for [MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.MetaData), [Table](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Table) and [Column](https://docs.sqlalchemy.org/en/20/core/metadata.html#sqlalchemy.schema.Column) is at [Describing Databases with MetaData](https://docs.sqlalchemy.org/en/20/core/metadata.html). The reference documentation for datatypes is at [SQL Datatype Objects](https://docs.sqlalchemy.org/en/20/core/types.html)

In [20]:
user_table.primary_key

PrimaryKeyConstraint(Column('id', Integer(), table=<user_account>, primary_key=True, nullable=False))

The constraint that is most typically declared explicitly is the ForeignKeyConstraint object that corresponds to a database foreign key constraint. When we declare tables that are related to each other, SQLAlchemy uses the presence of these foreign key constraint declarations not only so that they are emitted within CREATE statements to the database, but also to assist in constructing SQL expressions.

A ForeignKeyConstraint that involves only a single column on the target table is typically declared using a column-level shorthand notation via the ForeignKey object. Below we declare a second table address that will have a foreign key constraint referring to the user table:

In [21]:
from sqlalchemy import ForeignKey
address_table = Table(
    "address",
    metadata_obj,
    Column("id", Integer, primary_key=True),
    Column("user_id", ForeignKey("user_account.id"), nullable=False),
    Column("email_address", String, nullable=False),
)

When using the ForeignKey object within a Column definition, we can omit the datatype for that Column; it is automatically inferred from that of the related column, in the above example the Integer datatype of the user_account.id column.



# Emitting DDL to the Database

We've constructed an object structure that represents two database tables in a database, starting at the root MetaData object, then into two Table objects, each of which hold onto a collection of Column and Constraint objects. This object structure will be at the center of most operations we perform with both Core and ORM going forward.

The first useful thing we can do with this structure will be to emit CREATE TABLE statements or DDL, to our SQLite database so that we can insert and query data from them. We have already all the tools needed to do so, so by invoking the Metedata.create_all() method on our MetaData, sending it the Engine that refers to the target database:

In [22]:
metadata_obj.create_all(engine)

2026-01-20 16:06:45,232 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 16:06:45,235 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-20 16:06:45,236 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-20 16:06:45,239 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("user_account")
2026-01-20 16:06:45,240 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-20 16:06:45,242 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-20 16:06:45,243 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-20 16:06:45,245 INFO sqlalchemy.engine.Engine PRAGMA temp.table_info("address")
2026-01-20 16:06:45,247 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-20 16:06:45,250 INFO sqlalchemy.engine.Engine 
CREATE TABLE user_account (
	id INTEGER NOT NULL, 
	name VARCHAR(30), 
	fullname VARCHAR, 
	PRIMARY KEY (id)
)


2026-01-20 16:06:45,252 INFO sqlalchemy.engine.Engine [no key 0.00217s] ()
2026-01-20 16:06:45,254 INFO sqlalchemy.engine.Engine 
C

## Migration tools are usually appropriate

Overall, the CREATE / DROP feature of MetaData is useful for test suites, small and/or new applications, and applications that use short-lived databases. For management of an application database schema over the long term however, a schema management tool such as [Alembic](https://alembic.sqlalchemy.org/), which builds upon SQLAlchemy, is likely a better choice, as it can manage and orchestrate the process of incrementally altering a fixed database schema over time as the design of the application changes. 

When using the ORM, the process by which we declare Table metadata is usually combined with the process of declaring [mapped](https://docs.sqlalchemy.org/en/20/glossary.html#term-mapped) classes. The mapped class is any Python class we’d like to create, which will then have attributes on it that will be linked to the columns in a database table. While there are a few varieties of how this is achieved, the most common style is known as [declarative](https://docs.sqlalchemy.org/en/20/orm/declarative_config.html), and allows us to declare our user-defined classes and Table metadata at once.

# Establishing a Declarative Base

When using the ORM, the Metadata collection remains present, however it iteslf is associated with an ORM-only construct commonly referred towards as the **Declarative Base**. The most expedient way to acquire a new Declarative Base is to create a new class that subclasses the SQLAlchemy DeclarativeBase class:

In [23]:
from sqlalchemy.orm import DeclarativeBase
class Base(DeclarativeBase):
    pass

Above, the Base class is what we’ll call the Declarative Base. When we make new classes that are subclasses of Base, combined with appropriate class-level directives, they will each be established as a new ORM mapped class at class creation time, each one typically (but not exclusively) referring to a particular Table object.

The Declarative Base refers to a MetaData collection that is created for us automatically, assuming we didn’t provide one from the outside. This MetaData collection is accessible via the DeclarativeBase.metadata class-level attribute. As we create new mapped classes, they each will reference a Table within this MetaData collection:

In [24]:
Base.metadata

MetaData()

In [25]:
Base.registry

## Declaring Mapped Classes

With the `Base` class estblished, we can now define ORM mapped classes for the `user_account` and `address` tables in terms of new classes `User` and `Address`. We illustrate below the most modern form of Declarative, which is driven from [PEP 484](https://peps.python.org/pep-0484/) type annotations using a special type [Mapped](https://docs.sqlalchemy.org/en/20/orm/internals.html#sqlalchemy.orm.Mapped), which indicates attributes to be mapped as particular types:

In [26]:
from typing import List
from typing import Optional
from sqlalchemy.orm import Mapped
from sqlalchemy.orm import mapped_column
from sqlalchemy.orm import relationship

class User(Base):
    __tablename__ = "user_account"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(30))
    fullname: Mapped[Optional[str]]
    addresses: Mapped[List["Address"]] = relationship(back_populates="user")
    def __repr__(self) -> str:
        return f"User(id={self.id!r}, name={self.name!r}, fullname={self.fullname!r})"

class Address(Base):
    __tablename__ = "address"
    id: Mapped[int] = mapped_column(primary_key=True)
    email_address: Mapped[str]
    user_id = mapped_column(ForeignKey("user_account.id"))
    user: Mapped[User] = relationship(back_populates="addresses")
    def __repr__(self) -> str:
        return f"Address(id={self.id!r}, email_address={self.email_address!r})"







A user can have many email addresses,     an email address can not have many users.

* Two additional attributes, `User.addresses` and `Address.user` define a different kind of attribute called [relationship()](https://docs.sqlalchemy.org/en/20/orm/relationship_api.html#sqlalchemy.orm.relationship), which features similar annotation-aware configuration styles as shown. The [relationship()](https://docs.sqlalchemy.org/en/20/orm/relationship_api.html#sqlalchemy.orm.relationship) construct is discussed more fully at [Working with ORM Related Objects](https://docs.sqlalchemy.org/en/20/tutorial/orm_related_objects.html#tutorial-orm-related-objects)

* The classes are automatically given an `__init__` method if we don't declare one of our own. the default form of this method accepts all attribute names as optional keyword arguments.

In [27]:
sandy = User(name="sandy", fullname="Sandy Cheeks")

In [28]:
sandy

User(id=None, name='sandy', fullname='Sandy Cheeks')

https://docs.sqlalchemy.org/en/20/tutorial/orm_related_objects.html#tutorial-orm-related-objects

In [29]:
u1 = User(name="pkrabs", fullname="Pearl Krabs")
u1.addresses

[]

In [30]:
a1 = Address(email_address="pearl.krabs@gmail.com")
u1.addresses.append(a1)

In [31]:
u1.addresses

[Address(id=None, email_address='pearl.krabs@gmail.com')]

In [32]:
a1.user

User(id=None, name='pkrabs', fullname='Pearl Krabs')

In [33]:
a2 = Address(email_address="pearl@aol.com", user=u1)
u1.addresses

[Address(id=None, email_address='pearl.krabs@gmail.com'),
 Address(id=None, email_address='pearl@aol.com')]

In [34]:
type(session)

sqlalchemy.orm.session.Session

In [35]:
session.add(u1)
u1 in session

True

In [36]:
a1 in session


True

In [37]:
a2 in session

True

The three objects are now in the pending state; this means they are ready to be the subject of an INSERT operation but this has not yet proceeded; all three objects have no primary key assigned yet, and in addition, the a1 and a2 objects have an attribute called user_id which refers to the Column that has a ForeignKeyConstraint referring to the user_account.id column; these are also None as the objects are not yet associated with a real database row:

In [38]:
print(u1.id)

None


In [39]:
print(a1.user_id)

None


It’s at this stage that we can see the very great utility that the unit of work process provides; recall in the section INSERT usually generates the “values” clause automatically, rows were inserted into the user_account and address tables using some elaborate syntaxes in order to automatically associate the address.user_id columns with those of the user_account rows. Additionally, it was necessary that we emit INSERT for user_account rows first, before those of address, since rows in address are dependent on their parent row in user_account for a value in their user_id column.

When using the Session, all this tedium is handled for us and even the most die-hard SQL purist can benefit from automation of INSERT, UPDATE and DELETE statements. When we Session.commit() the transaction all steps invoke in the correct order, and furthermore the newly generated primary key of the user_account row is applied to the address.user_id column appropriately:



In [40]:
session.commit()

2026-01-20 17:07:36,201 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 17:07:36,203 INFO sqlalchemy.engine.Engine INSERT INTO user_account (name, fullname) VALUES (?, ?)
2026-01-20 17:07:36,204 INFO sqlalchemy.engine.Engine [generated in 0.00073s] ('pkrabs', 'Pearl Krabs')
2026-01-20 17:07:36,206 INFO sqlalchemy.engine.Engine INSERT INTO address (email_address, user_id) VALUES (?, ?) RETURNING id
2026-01-20 17:07:36,207 INFO sqlalchemy.engine.Engine [generated in 0.00012s (insertmanyvalues) 1/2 (ordered; batch not supported)] ('pearl.krabs@gmail.com', 1)
2026-01-20 17:07:36,208 INFO sqlalchemy.engine.Engine INSERT INTO address (email_address, user_id) VALUES (?, ?) RETURNING id
2026-01-20 17:07:36,208 INFO sqlalchemy.engine.Engine [insertmanyvalues 2/2 (ordered; batch not supported)] ('pearl@aol.com', 1)
2026-01-20 17:07:36,210 INFO sqlalchemy.engine.Engine COMMIT


In [41]:
u1.id

2026-01-20 17:08:36,023 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 17:08:36,027 INFO sqlalchemy.engine.Engine SELECT user_account.id AS user_account_id, user_account.name AS user_account_name, user_account.fullname AS user_account_fullname 
FROM user_account 
WHERE user_account.id = ?
2026-01-20 17:08:36,028 INFO sqlalchemy.engine.Engine [generated in 0.00145s] (1,)


1

In [42]:
u1.addresses

2026-01-20 17:09:12,761 INFO sqlalchemy.engine.Engine SELECT address.id AS address_id, address.email_address AS address_email_address, address.user_id AS address_user_id 
FROM address 
WHERE ? = address.user_id
2026-01-20 17:09:12,762 INFO sqlalchemy.engine.Engine [generated in 0.00129s] (1,)


[Address(id=1, email_address='pearl.krabs@gmail.com'),
 Address(id=2, email_address='pearl@aol.com')]

In [43]:
a1

Address(id=1, email_address='pearl.krabs@gmail.com')

In [44]:
a2

Address(id=2, email_address='pearl@aol.com')

In [45]:
a1.user

User(id=1, name='pkrabs', fullname='Pearl Krabs')

# Emitting DDL to the database from an ORM mapping



In [47]:
engine

Engine(sqlite+pysqlite:///:memory:)

In [48]:
Base.metadata.create_all(engine)

2026-01-20 17:16:14,620 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-01-20 17:16:14,621 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("user_account")
2026-01-20 17:16:14,622 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-20 17:16:14,623 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("address")
2026-01-20 17:16:14,624 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-01-20 17:16:14,625 INFO sqlalchemy.engine.Engine COMMIT


In this case, PRAGMA statements are run, but no new tables are generated since they are found to be present already: